# Retrieval Methods Comparison

Compare all Colette retrieval modes on the same questions side-by-side:

| Mode | Images | BM25 text | hindex nav | Name |
|------|--------|-----------|------------|------|
| `embedding_retrieval` | ✓ | — | — | Embedding |
| `text_search_retrieval` | — | ✓ | — | Text Search |
| `hybrid` | ✓ | ✓ | — | Hybrid |
| `vision_hierarchical` | ✓ | — | ✓ | hindex only |
| `embedding_retrieval + enable_hindex` | ✓ | — | ✓ | Embed+hindex |
| `text_search_retrieval + enable_hindex` | — | ✓ | ✓ | Text+hindex |
| `hybrid + enable_hindex` | ✓ | ✓ | ✓ | Hybrid+hindex |

**Sections:**
1. Runtime options
2. Service setup
3. Indexing (skip if already done)
4. Test questions & methods
5. Run all queries
6. Retrieved images
7. Retrieval confidence (distances & scores)
8. Page coverage heatmap
9. Method agreement (Jaccard)
10. Latency comparison
11. Answer comparison
12. hindex navigation details
13. Optional: manual relevance annotation → Precision@K / Recall@K


## 1. Runtime Options

In [ ]:
# ── Runtime options ───────────────────────────────────────────────────────────
TOP_K          = 4     # images to retrieve per method
DISPLAY_TOP_N  = 3     # images shown in the visual comparison grid (≤ TOP_K)
N_QUESTIONS    = None  # None = run all questions; integer to limit for quick tests


## 2. Service Setup

In [ ]:
import json, os, sys, time, base64, re, warnings
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "png"
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore", category=DeprecationWarning)

colette_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(colette_root, "src"))

from colette.jsonapi import JSONApi
from colette.apidata import APIData

print("Packages loaded.")


In [ ]:
app_dir       = os.path.join(colette_root, "app_colette")
models_dir    = os.path.join(colette_root, "models")
documents_dir = os.path.join(colette_root, "docs/pdf")

_saved = os.path.join(app_dir, "config.json")
config_file = _saved if os.path.exists(_saved) else os.path.join(
    colette_root, "src/colette/config/vrag_default.json")

app_name    = "app_compare"
colette_api = JSONApi()

print(f"app_dir      : {app_dir}")
print(f"documents_dir: {documents_dir}")
print(f"config_file  : {config_file}")


In [ ]:
# Re-create the service (safe to run multiple times in a session)
try:
    await colette_api.service_delete(app_name)
except Exception:
    pass

with open(config_file) as f:
    create_config = json.load(f)
create_config["app"]["repository"]        = app_dir
create_config["app"]["models_repository"] = models_dir

colette_api.service_create(app_name, APIData(**create_config))
print("Service created.")


## 3. Indexing (skip if already done)

All five methods share the same index.  Run **3a** once to build the hybrid
vector+BM25 index, and **3b** once to build the hindex hierarchy files.
After that re-indexing is never needed (unless documents change).


In [ ]:
# Check current index state
from colette.backends.hf.hindex import DocumentHierarchy
import json as _json

hindex_dir   = Path(app_dir) / "mm_index" / "hindex"
hindex_files = sorted(hindex_dir.glob("*.json")) if hindex_dir.exists() else []
mm_index_dir = Path(app_dir) / "mm_index"

print(f"mm_index exists  : {mm_index_dir.exists()}")
print(f"hindex JSON files: {len(hindex_files)}")
for p in hindex_files:
    h = DocumentHierarchy.from_dict(_json.loads(p.read_text(encoding="utf-8")))
    print(f"  {Path(h.doc_name).name}  pages={h.page_count}  topics={len(h.topics)}")


In [ ]:
# 3a — Build hybrid index (vector embeddings + BM25 text search)
# Uncomment and run once if mm_index doesn't exist yet.
#
# _cfg_hybrid = {
#     "parameters": {
#         "input": {
#             "data": [documents_dir],
#             "rag": {"retrieval_mode": "hybrid", "reindex": True},
#         }
#     }
# }
# colette_api.service_index(app_name, APIData(**_cfg_hybrid))
# print("Hybrid index built.")


In [ ]:
# 3b — Build hindex (topic hierarchy — does NOT touch the vector/BM25 index)
# Uncomment and run once if hindex JSON files don't exist yet.
#
# _cfg_hindex = {
#     "parameters": {
#         "input": {
#             "data": [documents_dir],
#             "rag": {
#                 "retrieval_mode": "vision_hierarchical",
#                 "update_index": True,
#                 "reindex": False,
#             },
#         }
#     }
# }
# colette_api.service_index(app_name, APIData(**_cfg_hindex))
# print("hindex built.")


## 4. Test Questions and Methods

In [ ]:
# ── Edit these to match your documents ───────────────────────────────────────
QUESTIONS = [
    # --- Core questions ---
    # 1. Tests topic-level navigation: a clear subject that maps to one topic cluster.
    #    hindex should shine; BM25 will hit on "errors" but may scatter across pages.
    "What are the identified sources of errors?",

    # 2. Tests structural/architectural recall: information spread across the pipeline
    #    description. Embedding should do well; hindex needs the right topic selected.
    "How does the visual RAG pipeline work?",

    # 3. Tests list extraction: a specific enumerable set buried in one section.
    #    BM25 wins if the exact phrase appears; embedding handles paraphrases.
    "What are the evaluation metrics used?",

    # --- Edge cases ---
    # 4. Semantic paraphrase stress-test: same meaning as Q1, different vocabulary.
    #    BM25 will likely miss "mistakes" / "fail" if the doc uses "errors" / "limitations".
    #    Embedding should bridge the lexical gap; hindex depends on topic label wording.
    "What kinds of mistakes does the system make and where does it tend to fail?",

    # 5. Cross-topic synthesis: answer requires reading across multiple topic clusters.
    #    Single-topic hindex navigation will struggle; hybrid/embedding should do better.
    "How do the reported errors relate to the evaluation results?",

    # 6. Visual-content-specific: the answer lives in a figure/diagram, not in text.
    #    Pure BM25 has no visual signal. Embedding (visual) and hindex (page context)
    #    should outperform text-only methods.
    "What does the main architecture or system diagram illustrate?",

    # 7. Out-of-scope probe: tests hallucination resistance.
    #    No method should confidently answer; observe which ones hallucinate vs. abstain.
    "What GPU hardware or compute cluster was used to run the experiments?",
]

# ── Methods ───────────────────────────────────────────────────────────────────
METHODS = [
    {
        "name":   "embedding",
        "label":  "Embedding",
        "color":  "#2196F3",
        "config": {"retrieval_mode": "embedding_retrieval"},
    },
    {
        "name":   "text_search",
        "label":  "Text Search",
        "color":  "#FF9800",
        "config": {"retrieval_mode": "text_search_retrieval"},
    },
    {
        "name":   "hybrid",
        "label":  "Hybrid",
        "color":  "#4CAF50",
        "config": {"retrieval_mode": "hybrid"},
    },
    # Pure hindex navigation — no embedding ranking, no BM25.
    # Hierarchy topic selection drives everything.
    {
        "name":   "vision_hier",
        "label":  "hindex only",
        "color":  "#9C27B0",
        "config": {"retrieval_mode": "vision_hierarchical"},
    },
    # Embedding k-NN + hindex navigation merged.
    # Tests whether hindex adds value on top of semantic search (no BM25).
    {
        "name":   "embed_hindex",
        "label":  "Embed+hindex",
        "color":  "#00BCD4",
        "config": {"retrieval_mode": "embedding_retrieval", "hindex": {"enable_hindex": True}},
    },
    # BM25 keyword search + hindex navigation merged.
    # Tests whether hindex adds value on top of lexical search (no embedding).
    {
        "name":   "text_hindex",
        "label":  "Text+hindex",
        "color":  "#795548",
        "config": {"retrieval_mode": "text_search_retrieval", "hindex": {"enable_hindex": True}},
    },
    # Full fusion: embedding + BM25 + hindex all merged.
    # The kitchen-sink baseline — should be hardest to beat but also slowest.
    {
        "name":   "hybrid_hindex",
        "label":  "Hybrid+hindex",
        "color":  "#F44336",
        "config": {"retrieval_mode": "hybrid", "hindex": {"enable_hindex": True}},
    },
]

qs = QUESTIONS[:N_QUESTIONS] if N_QUESTIONS else QUESTIONS
total_calls = len(qs) * len(METHODS)
print(f"Questions : {len(qs)}")
print(f"Methods   : {len(METHODS)}")
print(f"LLM calls : {total_calls}  (rough estimate: {total_calls * 40 // 60}–{total_calls * 70 // 60} min)")


## 5. Run All Queries

Runs every method × question combination and caches results in `ALL_RESULTS`.
Re-run individual methods by calling `run_query()` directly.


In [ ]:
def _decode_image(data_uri: str) -> Image.Image | None:
    try:
        b64 = re.sub(r"^data:image/.+;base64,", "", data_uri)
        return Image.open(BytesIO(base64.b64decode(b64)))
    except Exception:
        return None


def run_query(method: dict, question: str, top_k: int = 4) -> dict:
    rag_params = dict(method["config"])
    rag_params["top_k"] = top_k
    payload = {"parameters": {"input": {"message": question, "rag": rag_params}}}
    t0 = time.time()
    response = colette_api.service_predict(app_name, APIData(**payload))
    elapsed = round(time.time() - t0, 2)

    image_hits = []
    for item in response.sources.get("context", []):
        image_hits.append({
            "key":              item.get("key", ""),
            "page_number":      item.get("page_number"),
            "distance":         item.get("distance", float("nan")),
            "crop_label":       item.get("crop_label", ""),
            "retrieval_source": item.get("retrieval_source", "embedding"),
            "summary":          item.get("summary", ""),
            "image":            _decode_image(item.get("content", "")),
        })

    text_hits = [
        {
            "source":      h.get("source", ""),
            "page_number": h.get("page_number"),
            "score":       h.get("score", float("nan")),
            "content":     h.get("content", ""),
            "crop_label":  h.get("crop_label", ""),
        }
        for h in response.sources.get("text_context", [])
    ]

    return {
        "method":      method["name"],
        "question":    question,
        "answer":      response.output or "",
        "latency_s":   elapsed,
        "image_hits":  image_hits,
        "text_hits":   text_hits,
    }


In [ ]:
ALL_RESULTS: dict[tuple[str, str], dict] = {}

total = len(qs) * len(METHODS)
done  = 0
for q in qs:
    for m in METHODS:
        done += 1
        print(f"[{done}/{total}] {m['label']:<14} {q[:55]}…", end=" ", flush=True)
        try:
            r = run_query(m, q, top_k=TOP_K)
            ALL_RESULTS[(m["name"], q)] = r
            print(f"✓  {r['latency_s']:.1f}s  img={len(r['image_hits'])}  txt={len(r['text_hits'])}")
        except Exception as exc:
            print(f"ERROR: {exc}")

print(f"\nDone. {len(ALL_RESULTS)}/{total} successful.")


## 6. Retrieved Images

Top images returned per method for each question.  Page number, crop type, and
embedding distance are shown in the title.  Purple = hindex-sourced item,
blue = embedding-sourced item.


In [ ]:
N_IMG = min(DISPLAY_TOP_N, TOP_K)
visual_methods = [m for m in METHODS if m["name"] != "text_search"]

for q in qs:
    print(f"\n{'═' * 72}")
    print(f"Q: {q}")
    print("═" * 72)

    n_rows, n_cols = len(visual_methods), N_IMG
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.2 * n_rows),
                             squeeze=False)

    for row_i, m in enumerate(visual_methods):
        r = ALL_RESULTS.get((m["name"], q))
        hits = r["image_hits"] if r else []

        axes[row_i][0].set_ylabel(m["label"], fontsize=8, rotation=90, labelpad=45,
                                   color=m["color"], fontweight="bold")
        for col_i in range(n_cols):
            ax = axes[row_i][col_i]
            ax.axis("off")
            if col_i < len(hits):
                hit = hits[col_i]
                if hit["image"]:
                    ax.imshow(hit["image"])
                page = hit.get("page_number", "?")
                label = hit.get("crop_label", "")
                dist  = hit.get("distance", float("nan"))
                src   = hit.get("retrieval_source", "embedding")
                title_color = "#9C27B0" if src == "hindex" else "#1565C0"
                ax.set_title(f"p{page} | {label} | d={dist:.3f}",
                             fontsize=7, color=title_color)
            else:
                ax.text(0.5, 0.5, "—", ha="center", va="center",
                        transform=ax.transAxes, fontsize=16, color="#ccc")

    plt.suptitle(q, fontsize=9, y=1.01)
    plt.tight_layout()
    plt.show()

    # BM25 text snippets for any method that has text hits
    for m in METHODS:
        r = ALL_RESULTS.get((m["name"], q))
        if r and r["text_hits"]:
            print(f"\n{m['label']} — BM25 text hits:")
            for i, h in enumerate(r["text_hits"][:N_IMG]):
                snippet = h["content"][:120].replace("\n", " ")
                print(f"  [{i+1}] p{h['page_number']}  score={h['score']:.3f}  {snippet}")


## 7. Retrieval Confidence

**Embedding distance** — lower means the model considers the image closer to the
query.  **Text search score** — higher means stronger BM25 keyword match.


In [ ]:
# Methods that use BM25 — their signal lives in text_hits
BM25_METHODS = {"text_search", "text_hindex"}
# Methods that use embedding — only count hits where retrieval_source != "hindex"
# (hindex hits always have distance=0.0 and would distort the mean)
EMBED_METHODS = {"embedding", "hybrid", "vision_hier", "embed_hindex", "hybrid_hindex"}

dist_rows = []
for m in METHODS:
    for q in qs:
        r = ALL_RESULTS.get((m["name"], q))
        if not r:
            continue
        if m["name"] in BM25_METHODS:
            for rank, h in enumerate(r["text_hits"], 1):
                dist_rows.append({"method": m["label"], "q": q[:40],
                                  "rank": rank, "score": h["score"], "type": "BM25 score"})
        if m["name"] in EMBED_METHODS:
            embed_hits = [h for h in r["image_hits"]
                          if h.get("retrieval_source", "embedding") != "hindex"]
            for rank, h in enumerate(embed_hits, 1):
                dist_rows.append({"method": m["label"], "q": q[:40],
                                  "rank": rank, "score": h["distance"], "type": "embed distance"})

if dist_rows:
    df_score = pd.DataFrame(dist_rows)
    embed_method_labels = [m["label"] for m in METHODS if m["name"] in EMBED_METHODS]
    bm25_method_labels  = [m["label"] for m in METHODS if m["name"] in BM25_METHODS]

    df_embed = df_score[df_score["type"] == "embed distance"]
    if not df_embed.empty:
        fig = px.bar(
            df_embed.groupby(["method", "q"])["score"].mean().reset_index(),
            x="method", y="score", color="method", barmode="group", facet_col="q",
            title="Mean Embedding Distance per Method — embedding hits only (lower = more confident)",
            labels={"score": "Distance", "method": ""},
            category_orders={"method": embed_method_labels},
        )
        fig.update_layout(height=380, showlegend=False)
        fig.show()

    df_bm25 = df_score[df_score["type"] == "BM25 score"]
    if not df_bm25.empty:
        print("\nBM25 scores (methods with text search component):")
        display(df_bm25[["method", "q", "rank", "score"]]
                .rename(columns={"q": "question", "score": "BM25 score"}))


## 8. Page Coverage

Which pages does each method retrieve for a given question?
Blue = retrieved, white = not retrieved.


In [ ]:
# Determine total page count from hindex or from result data
hindex_files = sorted((Path(app_dir) / "mm_index" / "hindex").glob("*.json")) \
    if (Path(app_dir) / "mm_index" / "hindex").exists() else []
if hindex_files:
    _h0 = DocumentHierarchy.from_dict(_json.loads(hindex_files[0].read_text()))
    doc_page_count = _h0.page_count
else:
    all_pages = [
        hit["page_number"]
        for r in ALL_RESULTS.values()
        for hit in r["image_hits"] + r["text_hits"]
        if hit.get("page_number")
    ]
    doc_page_count = max(all_pages, default=20)

print(f"Document page count: {doc_page_count}")

method_labels = [m["label"] for m in METHODS]

# Methods whose full page coverage comes from both image and text channels
DUAL_CHANNEL = {"text_hindex"}

for q in qs:
    matrix = np.zeros((len(METHODS), doc_page_count), dtype=int)
    for row_i, m in enumerate(METHODS):
        r = ALL_RESULTS.get((m["name"], q))
        if not r:
            continue
        if m["name"] in DUAL_CHANNEL:
            hits = r["image_hits"] + r["text_hits"]
        elif m["name"] == "text_search":
            hits = r["text_hits"]
        else:
            hits = r["image_hits"]
        for h in hits:
            p = h.get("page_number")
            if p and 1 <= p <= doc_page_count:
                matrix[row_i, p - 1] = 1

    fig, ax = plt.subplots(figsize=(max(8, doc_page_count * 0.55), 2.4))
    im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(doc_page_count))
    ax.set_xticklabels([f"p{i+1}" for i in range(doc_page_count)], fontsize=7)
    ax.set_yticks(range(len(METHODS)))
    ax.set_yticklabels(method_labels, fontsize=8)
    ax.set_title(f"Page Coverage — {q[:65]}", fontsize=9)
    plt.tight_layout()
    plt.show()


## 9. Method Agreement

Jaccard similarity between each pair of methods on the **set of pages** they retrieve.
`1.0` = identical page sets, `0.0` = no overlap.  High off-diagonal values mean
those methods are redundant; low values mean they are complementary.


In [ ]:
def _jaccard(a: set, b: set) -> float:
    u = a | b
    return len(a & b) / len(u) if u else 1.0

method_labels = [m["label"] for m in METHODS]
n = len(METHODS)
DUAL_CHANNEL = {"text_hindex"}

for q in qs:
    page_sets = []
    for m in METHODS:
        r = ALL_RESULTS.get((m["name"], q))
        if not r:
            page_sets.append(frozenset())
            continue
        if m["name"] in DUAL_CHANNEL:
            hits = r["image_hits"] + r["text_hits"]
        elif m["name"] == "text_search":
            hits = r["text_hits"]
        else:
            hits = r["image_hits"]
        page_sets.append(frozenset(h["page_number"] for h in hits if h.get("page_number")))

    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            mat[i, j] = _jaccard(page_sets[i], page_sets[j])

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(method_labels, rotation=35, ha="right", fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(method_labels, fontsize=7)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if mat[i,j] > 0.6 else "black")
    ax.set_title(f"Method Agreement (Jaccard on retrieved pages)\n{q[:60]}", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


## 10. Latency

Total wall-clock time per call (retrieval + LLM generation).
`vision_hierarchical` and `hybrid+hindex` include an extra VLM navigation call.


In [ ]:
method_labels = [m["label"] for m in METHODS]

lat_rows = [
    {"method": m["label"], "question": q[:40], "latency_s": r["latency_s"],
     "color": m["color"]}
    for m in METHODS
    for q in qs
    if (r := ALL_RESULTS.get((m["name"], q)))
]

if lat_rows:
    df_lat = pd.DataFrame(lat_rows)

    fig = px.bar(
        df_lat, x="method", y="latency_s", color="method",
        barmode="group", facet_col="question",
        title="Total Latency per Method (retrieval + LLM generation)",
        labels={"latency_s": "Seconds", "method": ""},
        category_orders={"method": method_labels},
    )
    fig.update_layout(height=380, showlegend=False)
    fig.show()

    df_summary = df_lat.groupby("method")["latency_s"].agg(["mean", "min", "max"]).round(1)
    df_summary.columns = ["Mean (s)", "Min (s)", "Max (s)"]
    print("Summary:")
    display(df_summary)


## 11. Answer Comparison

LLM answer per method, truncated to 600 characters for readability.


In [ ]:
MAX_ANSWER_CHARS = 600

for q in qs:
    print(f"\n{'═' * 80}")
    print(f"Q: {q}")
    print("═" * 80)
    for m in METHODS:
        r = ALL_RESULTS.get((m["name"], q))
        answer = (r["answer"] if r else "(no result)") or "(empty)"
        words = len(answer.split())
        truncated = len(answer) > MAX_ANSWER_CHARS
        print(f"\n── {m['label']}  ({words} words, {r['latency_s'] if r else '?'}s) {'─' * 30}")
        print(answer[:MAX_ANSWER_CHARS] + ("…" if truncated else ""))


## 12. hindex Navigation Details

For `vision_hierarchical` and `hybrid+hindex`: shows which items came from the
hierarchy navigator vs. the vector embeddings, and any page-level VLM metadata
attached to hindex results.


In [ ]:
hindex_mode_names = {"vision_hier", "embed_hindex", "text_hindex", "hybrid_hindex"}

for q in qs:
    print(f"\nQ: {q[:70]}")
    for m in METHODS:
        if m["name"] not in hindex_mode_names:
            continue
        r = ALL_RESULTS.get((m["name"], q))
        if not r:
            print(f"  {m['label']}: (no result)")
            continue
        hi_hits = [h for h in r["image_hits"] if h.get("retrieval_source") == "hindex"]
        em_hits = [h for h in r["image_hits"] if h.get("retrieval_source") != "hindex"]
        bm25_hits = r["text_hits"]
        hi_pages   = sorted({h["page_number"] for h in hi_hits  if h.get("page_number")})
        em_pages   = sorted({h["page_number"] for h in em_hits  if h.get("page_number")})
        bm25_pages = sorted({h["page_number"] for h in bm25_hits if h.get("page_number")})
        print(f"  {m['label']}:")
        print(f"    hindex hits : {len(hi_hits):2d}  pages {hi_pages}")
        if em_hits:
            print(f"    embed  hits : {len(em_hits):2d}  pages {em_pages}")
        if bm25_hits:
            print(f"    BM25   hits : {len(bm25_hits):2d}  pages {bm25_pages}")
        for hit in hi_hits[:2]:
            if hit.get("summary"):
                print(f"    → p{hit['page_number']} summary: {hit['summary'][:100]}")

# Load topic info from hierarchy files for context
if hindex_files:
    print("\n── Topic structures ─────────────────────────────────────────────")
    for p in hindex_files[:2]:
        h = DocumentHierarchy.from_dict(_json.loads(p.read_text(encoding="utf-8")))
        print(f"\n{h.doc_name}")
        for t in h.topics:
            print(f"  [{t.topic_id}] {t.theme}  pages={t.page_indices}")


## 13. Optional: Manual Relevance Annotation

After reviewing section 6, mark each retrieved image as relevant (1) or not (0).
Key format: `(method_name, question_0based_index, rank_1based)`.

Running this cell computes **Precision@K** and **Recall@K** per method.


In [ ]:
method_labels = [m["label"] for m in METHODS]

# ── Fill in your judgements here ─────────────────────────────────────────────
# Keys: (method_name, question_index, rank)  →  1 = relevant, 0 = not relevant
# method_name must match the "name" field in METHODS (e.g. "embedding", "hybrid_hindex")
RELEVANCE: dict[tuple[str, int, int], int] = {
    # ("embedding",     0, 1): 1,   # Q1, rank-1 image is relevant
    # ("embedding",     0, 2): 0,   # Q1, rank-2 image is not relevant
    # ("hybrid_hindex", 0, 1): 1,
    # ("text_hindex",   0, 1): 1,   # text_hindex: image_hits are hindex pages
    # ...
}
# ─────────────────────────────────────────────────────────────────────────────

# text_search has no image hits; text_hindex image hits are hindex-sourced pages
TEXT_ONLY_METHODS = {"text_search"}

if not RELEVANCE:
    print("RELEVANCE is empty — add your annotations above and re-run this cell.")
else:
    pk_rows = []
    for m in METHODS:
        if m["name"] in TEXT_ONLY_METHODS:
            continue
        for q_idx, q in enumerate(qs):
            r = ALL_RESULTS.get((m["name"], q))
            if not r:
                continue
            hits = r["image_hits"]
            labels_for_hits = [RELEVANCE.get((m["name"], q_idx, k + 1), -1) for k in range(len(hits))]
            annotated = [x for x in labels_for_hits if x >= 0]
            if not annotated:
                continue
            n_rel_retrieved = sum(x for x in annotated)
            total_rel = sum(
                v for (mn, qi, _), v in RELEVANCE.items()
                if mn == m["name"] and qi == q_idx and v == 1
            )
            pk_rows.append({
                "method":        m["label"],
                "question":      q[:45],
                "P@K":           round(n_rel_retrieved / len(annotated), 3),
                "R@K":           round(n_rel_retrieved / total_rel, 3) if total_rel else float("nan"),
                "annotated":     len(annotated),
                "rel_retrieved": n_rel_retrieved,
            })

    if pk_rows:
        df_pk = pd.DataFrame(pk_rows)
        display(df_pk)
        fig = px.bar(df_pk, x="method", y="P@K", color="method", barmode="group",
                     facet_col="question", title="Precision@K per Method",
                     category_orders={"method": method_labels})
        fig.update_layout(height=380, showlegend=False)
        fig.show()
    else:
        print("No annotated items found — check key format in RELEVANCE dict.")


## Cleanup

Run before restarting the kernel to flush and close kvstore.

In [ ]:
await colette_api.service_delete(app_name)